# 04b — Step 5 Phase B: Random Forest + Gradient Boosting baselines

**Project:** Multimodal Deep Learning for Short-Term Blood Glucose Forecasting in T1D (HUPA-UCM).

Phase B trains the non-linear / ensemble references on the same flattened input as Phase A `(N, 24·17 + 16) = (N, 424)`. Two models:

1. **Random Forest** — `sklearn.ensemble.RandomForestRegressor`, native multi-output. Tests whether non-linear axis-aligned splits over the engineered feature set improve over the linear projection.
2. **HistGradientBoosting** — `sklearn.ensemble.HistGradientBoostingRegressor`, the sklearn-native equivalent of LightGBM (no extra install needed). One model per horizon (sklearn HistGB does not support multi-output natively). Early stopping on an internal 10 % validation slice of the training set; our external VAL split is never touched during fit.

Both use the same evaluation bundle, the same per-patient chronological split, and the same patient-averaged reporting convention as Phase A. The fitted models are persisted under `outputs/models/` for downstream reuse by the explainability and application notebooks.

**Compute:** CPU only. Full local run on 68 395 train / 45 382 val / 45 395 test samples is approximately 10–20 minutes wall-clock on a recent multi-core CPU (use `n_jobs=-1`). On Colab without GPU the same wall-clock applies. Subset to verify on a tiny budget with the `DEBUG = True` switch in cell 4.

**Optional XGBoost:** the next cell installs xgboost on Colab so the user can swap in `xgboost.XGBRegressor` for the GBM head if desired. The Phase B report uses sklearn HistGB so the run is reproducible without third-party installs.

## 0. Colab boilerplate

In [ ]:
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/glucose-thesis'
    BASE_PATH = DRIVE_PROJECT_PATH
except ImportError:
    IN_COLAB = False
    BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))

print('IN_COLAB =', IN_COLAB)
print('BASE_PATH =', BASE_PATH)

In [ ]:
# Optional — install xgboost on Colab if you want to swap it in for the GBM head.
# The default path uses sklearn HistGradientBoostingRegressor and needs no install.
if IN_COLAB:
    !pip install -q xgboost >/dev/null  # noqa

## 1. Setup, imports, debug switch

In [ ]:
src_path = os.path.join(BASE_PATH, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import time
from collections import defaultdict
import numpy as np
import pandas as pd

import config as C
from baselines import RandomForestBaseline, HistGBMBaseline
from evaluate import evaluate_model, compact_summary

np.random.seed(C.SEED)

TABLES_DIR = os.path.join(BASE_PATH, 'outputs', 'tables')
MODELS_DIR = os.path.join(BASE_PATH, 'outputs', 'models')
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Set DEBUG = True for a 5k-train / 2k-val / 2k-test smoke run with tiny model sizes.
DEBUG = False
print('DEBUG =', DEBUG)

## 2. Load sequences and split

Same `.npz` file as Phase A — the `split` mask carries the per-patient chronological 70/15/15 cut with an 18-step boundary buffer. No leakage protection needed in this notebook; it was enforced at sequence-construction time in Step 3.

In [ ]:
npz_path = os.path.join(BASE_PATH, C.SEQUENCES_NPZ)
d = np.load(npz_path, allow_pickle=True)

X_dyn  = d['X_dynamic'].astype(np.float32)
X_stat = d['X_static'].astype(np.float32)
y      = d['y'].astype(np.float32)
pid    = d['participant_ids'].astype(str)
split  = d['split'].astype(str)
feat_dyn  = [str(s) for s in d['feature_names_dynamic']]
feat_stat = [str(s) for s in d['feature_names_static']]

def slice_split(name):
    m = split == name
    return dict(X_dyn=X_dyn[m], X_stat=X_stat[m], y=y[m], pid=pid[m])

def subsample(sp, n_cap, seed=C.SEED):
    if n_cap <= 0 or len(sp['y']) <= n_cap:
        return sp
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(sp['y']), size=n_cap, replace=False))
    return {k: v[idx] for k, v in sp.items()}

train = slice_split('train')
val   = slice_split('val')
test  = slice_split('test')

if DEBUG:
    train = subsample(train, 5000); val = subsample(val, 2000); test = subsample(test, 2000)

print(f'shapes: X_dyn={X_dyn.shape}, X_stat={X_stat.shape}, y={y.shape}')
print(f'after DEBUG: train={len(train["y"])}, val={len(val["y"])}, test={len(test["y"])}')

rf_n_est, gbm_max_iter = (50, 100) if DEBUG else (300, 300)

## 3. Random Forest baseline

Hyperparameters were chosen as **evidence-based defaults** rather than via grid search, because the search space is small for this dataset scale and the validation MAE surface for RF on 424 collinear features is known to be flat for moderate changes in `n_estimators` and `max_depth`. The choices are:

- `n_estimators=300` — beyond ~200 the marginal gain plateaus for this `(N, F)` regime; 300 is the conservative side of the plateau.
- `max_depth=25` — caps tree growth to keep memory bounded and reduce overfit, while still allowing ~$2^{25}$ leaves which exceeds the train sample count.
- `min_samples_leaf=20` — regularises against splitting on noise in 5-min CGM data.
- `n_jobs=-1` — use all available CPU cores.
- `random_state=C.SEED` — full reproducibility.

Native multi-output: a single forest predicts all three horizons by storing a `(3,)` leaf vector and averaging across trees.

In [ ]:
t = time.time()
rf = RandomForestBaseline(
    n_estimators=rf_n_est,
    max_depth=25,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=C.SEED,
).fit(
    train['X_dyn'], train['X_stat'], train['y'],
    feature_names_dynamic=feat_dyn,
    feature_names_static=feat_stat,
)
print(f'RF fit time = {time.time() - t:.1f}s')

imp = rf.feature_importance_table(top_k=20)
imp.insert(0, 'model', 'rf')
imp.to_csv(os.path.join(TABLES_DIR, 'phase_b_rf_top_importance.csv'), index=False)
imp.head(15)

## 4. HistGradientBoosting baseline

Three independent `HistGradientBoostingRegressor` heads, one per horizon, because sklearn HistGB does not support multi-output regression natively. Each head uses **internal early stopping** on a 10 % slice carved from the training set — our external VAL split is never touched during the fit, preserving its role as a clean comparison surface.

Hyperparameters: `learning_rate=0.05`, `max_depth=8`, `min_samples_leaf=20`, `max_iter=300` with `n_iter_no_change=20`. The combination of moderate depth + low learning rate + early stopping is the standard guard against overfit on tabular regression.

In [ ]:
t = time.time()
gbm = HistGBMBaseline(
    max_iter=gbm_max_iter,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=20,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.1,
    random_state=C.SEED,
).fit(
    train['X_dyn'], train['X_stat'], train['y'],
    feature_names_dynamic=feat_dyn,
    feature_names_static=feat_stat,
)
print(f'GBM fit time = {time.time() - t:.1f}s')
print(f'iters used per horizon: {gbm.n_iters_used_}  (cap = {gbm_max_iter})')

pd.DataFrame({
    'horizon_min': list(C.HORIZON_MINUTES),
    'horizon_idx': list(range(len(C.HORIZON_MINUTES))),
    'n_iters_used': gbm.n_iters_used_,
    'max_iter_cap': gbm_max_iter,
}).to_csv(os.path.join(TABLES_DIR, 'phase_b_gbm_n_iters.csv'), index=False)

## 5. Persist models

In [ ]:
import joblib
joblib.dump(
    {'model': rf.model_, 'params': rf.params, 'feature_names': rf.feature_names_},
    os.path.join(MODELS_DIR, 'rf_phase_b.joblib'),
)
joblib.dump(
    {'models': gbm.models_, 'params': gbm.params,
     'n_iters_used': gbm.n_iters_used_, 'feature_names': gbm.feature_names_},
    os.path.join(MODELS_DIR, 'gbm_phase_b.joblib'),
)
print('saved rf_phase_b.joblib and gbm_phase_b.joblib')

## 6. Evaluate on VAL + TEST and aggregate

In [ ]:
bundles = []
for tag, model in [(f'rf_n{rf_n_est}', rf), (f'gbm_lr0.05_d8', gbm)]:
    for name, sp in [('val', val), ('test', test)]:
        yhat = model.predict(sp['X_dyn'], sp['X_stat'])
        bundles.append(evaluate_model(sp['y'], yhat, sp['pid'], tag, name))

name_map = {
    'per_horizon':       'phase_b_per_horizon.csv',
    'per_zone':          'phase_b_per_zone.csv',
    'per_patient':       'phase_b_per_patient.csv',
    'patient_averaged':  'phase_b_patient_averaged.csv',
    'clarke_eg':         'phase_b_clarke.csv',
}
by_key = defaultdict(list)
for b in bundles:
    for key, df in b.items():
        by_key[key].append(df)
for key, frames in by_key.items():
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(os.path.join(TABLES_DIR, name_map[key]), index=False)
    print(f'saved {name_map[key]:35s} rows={len(out)}')

compact = pd.concat([compact_summary(b) for b in bundles], ignore_index=True)
compact.to_csv(os.path.join(TABLES_DIR, 'phase_b_summary.csv'), index=False)
show = ['model','split','horizon_min','mae','rmse','mae_pat_avg','rmse_pat_avg','clarke_pct_A','clarke_pct_B','clarke_pct_D']
compact[[c for c in show if c in compact.columns]]

## 7. Per-zone comparison Phase A + Phase B

The clinically critical check. We concatenate the per-zone tables from both phases and pivot to compare hypo / TIR / hyper MAE side by side. If a Phase B model still loses to Persistence in the hypo zone, the conclusion is identical to Phase A — non-linearity alone does not solve the imbalanced-loss bias, and the hypo failure must be addressed in Phase C with a loss-function intervention rather than a model-architecture one.

In [ ]:
pa_zone = pd.read_csv(os.path.join(TABLES_DIR, 'phase_a_per_zone.csv'))
pb_zone = pd.read_csv(os.path.join(TABLES_DIR, 'phase_b_per_zone.csv'))
all_zone = pd.concat([pa_zone, pb_zone], ignore_index=True)
test_mae_zone = (all_zone[(all_zone.split=='test') & (all_zone.metric=='mae')]
                 .pivot_table(index=['model','horizon_min'], columns='zone', values='value')
                 .round(2))
print('Test MAE (mg/dL) by glycaemic zone — Phase A + Phase B:')
print(test_mae_zone)

## 8. Phase B conclusions and Phase C preview

Write the conclusions paragraph after the run completes, using the printed numbers above. The expected pattern, based on prior literature for tabular GBM on CGM data:

- **GBM should win pooled MAE at all horizons** because gradient boosting on engineered tabular features is consistently the strongest non-neural model class for tabular regression.
- **RF should improve over Ridge on pooled MAE** at 30–60 min but plateau at 90 min where the velocity-extrapolation signal saturates.
- **Both tree models may still lose to Persistence in the hypo zone**, because squared-loss boosting and forest splits are themselves biased toward the dense TIR mass.

**Next:** `notebooks/04c_phase_c_neural.ipynb` for Phase C — LSTM / GRU on the dynamic tensor with the static branch, then the proposed CNN-GRU with cross-attention fusion and asymmetric loss. Phase C is GPU-recommended (Colab T4 is sufficient).